# Botanix — Swin-T: PlantCLEF 2026 Pre-train → Multi-Label Ürün Hastalık Tespiti

**Mimari:** Swin Transformer Tiny (Swin-T, ~28M parametre)  
**PlantCLEF 2026:** https://www.imageclef.org/PlantCLEF2026  
**Hastalık Dataset:** https://www.kaggle.com/datasets/mertcangelbal/plant-leaf-disease-classification-dataset  
**GPU:** H200 NVL (Vast.ai) — önerilen

## Çalışma Akışı

| Faz | Dataset | Görev | Kayıp | Açıklama |
|-----|---------|-------|-------|----------|
| **Faz 1** | PlantCLEF 2024 (~1.4M görüntü, 7806 tür) | Single-label | CrossEntropy + LabelSmoothing | Botanik alan bilgisi |
| **Faz 2** | Hastalık veri seti (105 sınıf + Unknown) | **Multi-label** | BCEWithLogitsLoss | Ürün bazlı hastalık tespiti |

## Boyut Stratejisi

| Dataset | Orijinal | Model Girişi | Yöntem |
|---------|----------|-------------|--------|
| PlantCLEF | 800px (max side) | 224×224 | RandomResizedCrop(scale=0.4–1.0) |
| Hastalık seti | 256px | 224×224 | Resize(256) → RandomCrop(224) |

## Bilinmeyen Hastalık Çözümü

Model bir bitkiyi tanıdığı ama hastalığını bilmediği durumda üç katmanlı bir strateji uygulanır:

1. **Confidence Threshold:** Tüm sınıf olasılıkları < `unknown_threshold` ise → `Unknown Disease` döner  
2. **Entropy Uncertainty:** Çıkış dağılımı yüksek entropi gösteriyorsa → model kararsız kabul edilir  
3. **Healthy Sınıfı:** Eğitim verisine `Healthy` kategorisi eklenerek model sağlıklı bitki-hastalık ayrımı öğrenir

In [ ]:
# ── Kurulum & Dataset İndirme ─────────────────────────────────────────────────
!pip install -q kagglehub timm

import kagglehub, shutil, os, subprocess
from pathlib import Path

PLANTCLEF_DIR = Path("../plantclef")
FINETUNE_DIR  = Path("../data")

PLANTCLEF_URL = (
    "https://lab.plantnet.org/LifeCLEF/PlantCLEF2024/single_plant_training_data/"
    "PlantCLEF2024singleplanttrainingdata_800_max_side_size.tar"
)
PLANTCLEF_CSV_URL = (
    "https://lab.plantnet.org/LifeCLEF/PlantCLEF2024/single_plant_training_data/"
    "PlantCLEF2024singleplanttrainingdata.csv"
)
PLANTCLEF_TAR = PLANTCLEF_DIR / "PlantCLEF2024singleplanttrainingdata_800_max_side_size.tar"

# ── 1. PlantCLEF İndirme ──────────────────────────────────────────────────────
PLANTCLEF_DIR.mkdir(parents=True, exist_ok=True)

# Kaç görüntü klasörü var kontrol et (indirilip indirilmediğini anlamak için)
existing_classes = [d for d in PLANTCLEF_DIR.iterdir() if d.is_dir()] if PLANTCLEF_DIR.exists() else []

if len(existing_classes) > 100:
    print(f"PlantCLEF zaten mevcut: {len(existing_classes)} tür klasörü bulundu.")
else:
    print("PlantCLEF indiriliyor (~160GB) — bu işlem uzun sürer...")
    print(f"URL: {PLANTCLEF_URL}\n")

    # wget ile indirme (-c: kaldığı yerden devam eder)
    if not PLANTCLEF_TAR.exists():
        ret = subprocess.run([
            "wget", "-c", "-q", "--show-progress",
            PLANTCLEF_URL, "-P", str(PLANTCLEF_DIR)
        ])
        if ret.returncode != 0:
            print("HATA: wget başarısız. Terminalde manuel olarak çalıştır:")
            print(f"  wget -c '{PLANTCLEF_URL}' -P {PLANTCLEF_DIR}")
        else:
            print("İndirme tamamlandı.")
    else:
        print(f"Tar dosyası zaten mevcut: {PLANTCLEF_TAR}")

    # tar dosyası varsa çıkart
    if PLANTCLEF_TAR.exists():
        print("Tar arşivi çıkartılıyor...")
        ret = subprocess.run([
            "tar", "-xf", str(PLANTCLEF_TAR), "-C", str(PLANTCLEF_DIR)
        ])
        if ret.returncode == 0:
            print("Çıkartma tamamlandı.")
            # Disk alanı kazanmak için tar'ı sil (isteğe bağlı)
            # PLANTCLEF_TAR.unlink()
        else:
            print("HATA: tar çıkartma başarısız.")

    # CSV metadata indir
    csv_path = PLANTCLEF_DIR / "PlantCLEF2024singleplanttrainingdata.csv"
    if not csv_path.exists():
        subprocess.run(["wget", "-q", PLANTCLEF_CSV_URL, "-P", str(PLANTCLEF_DIR)])
        print(f"CSV indirildi: {csv_path}")

# ── 2. Hastalık Veri Seti (Fine-tune) ────────────────────────────────────────
_raw = kagglehub.dataset_download("mertcangelbal/plant-leaf-disease-classification-dataset")
print("\nHastalık dataset:", _raw)
if not FINETUNE_DIR.exists():
    shutil.copytree(_raw, str(FINETUNE_DIR))
    print(f"Kopyalandı → {FINETUNE_DIR}")
else:
    print(f"Mevcut → {FINETUNE_DIR}")

# ── Durum Özeti ───────────────────────────────────────────────────────────────
pt_classes = len([d for d in PLANTCLEF_DIR.iterdir() if d.is_dir()]) if PLANTCLEF_DIR.exists() else 0
print(f"\n{'='*45}")
print(f"PlantCLEF dizini : {PLANTCLEF_DIR.resolve()}")
print(f"PlantCLEF tür    : {pt_classes} klasör {'✅' if pt_classes > 100 else '❌ henüz hazır değil'}")
print(f"Fine-tune dizini : {FINETUNE_DIR.resolve()}")
print(f"Fine-tune mevcut : {'✅' if FINETUNE_DIR.exists() else '❌'}")
print(f"{'='*45}")

In [ ]:
# ── Kütüphaneler ─────────────────────────────────────────────────────────────
import os, time, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image
import timm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, average_precision_score
)
from sklearn.preprocessing import label_binarize

print(f"PyTorch  : {torch.__version__}")
print(f"timm     : {timm.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Konfigürasyon ─────────────────────────────────────────────────────────────
CONFIG = {
    # Dizinler
    "plantclef_dir":     "../plantclef",
    "finetune_dir":      "../data",

    # Görüntü
    "img_size":          224,
    "num_workers":       8,

    # Faz 1 — PlantCLEF Pre-train
    "plantclef_classes": 7806,
    "pretrain_epochs":   10,
    "pretrain_batch":    64,    # H200 için 64; RTX 5090 için 32
    "pretrain_lr":       3e-4,
    "pretrain_warmup":   2,
    "label_smoothing":   0.1,
    "pretrain_save":     "./checkpoints/swin_t_plantclef.pth",

    # Faz 2 — Multi-label Fine-tune
    "finetune_classes":  105,   # gerçek sınıf sayısı, aşağıda güncellenir
    "finetune_epochs":   30,
    "finetune_batch":    32,
    "finetune_lr":       5e-5,
    "freeze_epochs":     5,
    "weight_decay":      1e-4,
    "finetune_save":     "./checkpoints/swin_t_multilabel_best.pth",

    # Bilinmeyen hastalık stratejisi
    "ml_threshold":      0.5,   # sigmoid karar eşiği
    "unknown_threshold": 0.3,   # tüm olasılıklar bu altında → Unknown
    "entropy_threshold": 0.85,  # normalize entropy bu üstünde → Uncertain

    "model_name":        "swin_t_multilabel",
}

os.makedirs("./checkpoints", exist_ok=True)
os.makedirs("./results", exist_ok=True)
print("Konfigürasyon yüklendi.")
print(f"Pre-train : {CONFIG['pretrain_epochs']} epoch | batch {CONFIG['pretrain_batch']}")
print(f"Fine-tune : {CONFIG['finetune_epochs']} epoch | batch {CONFIG['finetune_batch']}")

In [ ]:
# ── Veri Dönüşümleri — Boyut Uyumu ────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
S    = CONFIG["img_size"]

# PlantCLEF: 800px → 224 (RandomResizedCrop ile geniş alan öğrenimi)
pretrain_transforms = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.4, 1.0), ratio=(0.75, 1.33),
                                 interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Hastalık seti: 256px → 224 (hastalık bölgesini koruyarak kırp)
finetune_train_tf = transforms.Compose([
    transforms.Resize((S + 32, S + 32), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(S),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
    transforms.RandomGrayscale(p=0.03),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

finetune_eval_tf = transforms.Compose([
    transforms.Resize((S, S), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("Dönüşümler hazır.")
print(f"  PlantCLEF (800px) → RandomResizedCrop({S}) | scale 0.4-1.0")
print(f"  Hastalık  (256px) → Resize({S+32}) → RandomCrop({S})")

In [ ]:
# ── DataLoader ────────────────────────────────────────────────────────────────
plantclef_path = Path(CONFIG["plantclef_dir"])
finetune_path  = Path(CONFIG["finetune_dir"])

# ── Faz 1: PlantCLEF ─────────────────────────────────────────────────────────
pretrain_loader = None
if plantclef_path.exists():
    _pt_root = plantclef_path / "train" if (plantclef_path / "train").exists() else plantclef_path
    pretrain_dataset = datasets.ImageFolder(_pt_root, transform=pretrain_transforms)
    CONFIG["plantclef_classes"] = len(pretrain_dataset.classes)
    pretrain_loader = DataLoader(
        pretrain_dataset, batch_size=CONFIG["pretrain_batch"],
        shuffle=True, num_workers=CONFIG["num_workers"],
        pin_memory=True, drop_last=True,
    )
    print(f"PlantCLEF: {len(pretrain_dataset):,} görüntü | {CONFIG['plantclef_classes']} tür")
else:
    print("PlantCLEF bulunamadı — Faz 1 atlanacak.")

# ── Faz 2: Hastalık Veri Seti ─────────────────────────────────────────────────
ft_train_raw = datasets.ImageFolder(finetune_path / "train", transform=finetune_train_tf)
ft_val_raw   = datasets.ImageFolder(finetune_path / "val",   transform=finetune_eval_tf)
ft_test_raw  = datasets.ImageFolder(finetune_path / "test",  transform=finetune_eval_tf)

class_names = ft_train_raw.classes
CONFIG["finetune_classes"] = len(class_names)
print(f"\nHastalık seti: Train {len(ft_train_raw):,} | Val {len(ft_val_raw):,} | Test {len(ft_test_raw):,}")
print(f"Sınıf sayısı : {CONFIG['finetune_classes']}")

# ── Multi-label Dataset Wrapper ───────────────────────────────────────────────
# ImageFolder single-label indeksi → one-hot float tensör
class MultiLabelWrapper(Dataset):
    def __init__(self, base, num_classes):
        self.base = base
        self.num_classes = num_classes
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, label = self.base[idx]
        target = torch.zeros(self.num_classes, dtype=torch.float32)
        target[label] = 1.0
        return img, target

ft_train = MultiLabelWrapper(ft_train_raw, CONFIG["finetune_classes"])
ft_val   = MultiLabelWrapper(ft_val_raw,   CONFIG["finetune_classes"])
ft_test  = MultiLabelWrapper(ft_test_raw,  CONFIG["finetune_classes"])

ft_train_loader = DataLoader(ft_train, batch_size=CONFIG["finetune_batch"],
                             shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
ft_val_loader   = DataLoader(ft_val,   batch_size=CONFIG["finetune_batch"],
                             shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
ft_test_loader  = DataLoader(ft_test,  batch_size=CONFIG["finetune_batch"],
                             shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

# Sınıf ağırlıkları — BCEWithLogitsLoss pos_weight
ft_counts = np.array([
    max(len(list((finetune_path / "train" / c).glob("*"))), 1)
    for c in class_names
], dtype=np.float32)
total_n = ft_counts.sum()
pos_weight = torch.tensor((total_n - ft_counts) / ft_counts, dtype=torch.float32).to(DEVICE)
print(f"pos_weight: shape={pos_weight.shape} min={pos_weight.min():.2f} max={pos_weight.max():.2f}")

In [ ]:
# ── Model — Swin-T ────────────────────────────────────────────────────────────
class BotanixSwinT(nn.Module):
    """
    Swin-T tabanlı multi-label ürün hastalık tespit modeli.
    Faz 1: num_classes=7806 (PlantCLEF, CrossEntropy)
    Faz 2: num_classes=105  (hastalık, BCEWithLogitsLoss)
    """
    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )
        self.feat_dim = self.backbone.num_features  # 768
        self.head = nn.Sequential(
            nn.LayerNorm(self.feat_dim),
            nn.Dropout(p=0.3),
            nn.Linear(self.feat_dim, num_classes),
        )
        self._init_head()

    def _init_head(self):
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def replace_head(self, num_classes: int):
        self.head = nn.Sequential(
            nn.LayerNorm(self.feat_dim),
            nn.Dropout(p=0.3),
            nn.Linear(self.feat_dim, num_classes),
        )
        self._init_head()
        self.head = self.head.to(next(self.backbone.parameters()).device)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

model = BotanixSwinT(num_classes=CONFIG["plantclef_classes"], pretrained=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre  : {total_params:,}")
print(f"Model boyutu (est): {total_params * 4 / 1e6:.1f} MB")

In [ ]:
# ── Faz 1: Optimizer & Scheduler ─────────────────────────────────────────────
pretrain_criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

pretrain_optimizer = optim.AdamW([
    {"params": model.backbone.parameters(), "lr": CONFIG["pretrain_lr"]},
    {"params": model.head.parameters(),     "lr": CONFIG["pretrain_lr"] * 2},
], weight_decay=CONFIG["weight_decay"])

def warmup_cosine(epoch, warmup, total):
    if epoch < warmup:
        return (epoch + 1) / warmup
    p = (epoch - warmup) / max(total - warmup, 1)
    return 0.5 * (1.0 + np.cos(np.pi * p))

pretrain_scheduler = optim.lr_scheduler.LambdaLR(
    pretrain_optimizer,
    lr_lambda=lambda e: warmup_cosine(e, CONFIG["pretrain_warmup"], CONFIG["pretrain_epochs"])
)
print("Faz 1 optimizer hazır.")

In [ ]:
# ── Yardımcı Fonksiyonlar ─────────────────────────────────────────────────────
def train_epoch_sl(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_sl(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def train_epoch_ml(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total = 0.0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)
    return total_loss / total

@torch.no_grad()
def eval_ml(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss, total = 0.0, 0
    all_preds, all_targets = [], []
    for imgs, targets in loader:
        imgs, targets = imgs.to(device), targets.to(device)
        logits = model(imgs)
        loss = criterion(logits, targets)
        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)
        preds = (torch.sigmoid(logits) >= threshold).float()
        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())
    P = torch.cat(all_preds).numpy()
    T = torch.cat(all_targets).numpy()
    f1_mi = f1_score(T, P, average="micro", zero_division=0)
    f1_ma = f1_score(T, P, average="macro", zero_division=0)
    return total_loss / total, f1_mi, f1_ma

print("Yardımcı fonksiyonlar hazır.")

In [ ]:
# ── FAZ 1: PlantCLEF Pre-train ────────────────────────────────────────────────
history_p1 = {"train_loss": [], "train_acc": [], "lr": []}
total_pretrain_time = 0

if pretrain_loader is not None:
    print("=" * 65)
    print("FAZ 1 — PlantCLEF Pre-train (Swin-T)")
    print(f"Dataset: {len(pretrain_dataset):,} görüntü | {CONFIG['plantclef_classes']} tür")
    print("=" * 65)
    t_start = time.time()

    for epoch in range(1, CONFIG["pretrain_epochs"] + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch_sl(
            model, pretrain_loader, pretrain_optimizer, pretrain_criterion, DEVICE
        )
        lr_now = pretrain_optimizer.param_groups[0]["lr"]
        pretrain_scheduler.step()

        history_p1["train_loss"].append(tr_loss)
        history_p1["train_acc"].append(tr_acc)
        history_p1["lr"].append(lr_now)

        elapsed = time.time() - t0
        eta = (CONFIG["pretrain_epochs"] - epoch) * elapsed / 60
        print(f"[Pre-train] Epoch [{epoch:02d}/{CONFIG['pretrain_epochs']}] "
              f"Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
              f"LR: {lr_now:.2e} | {elapsed:.0f}s | ETA: {eta:.0f}dk")

    total_pretrain_time = time.time() - t_start
    torch.save(model.state_dict(), CONFIG["pretrain_save"])
    print(f"\nPre-train tamamlandı : {total_pretrain_time/60:.1f} dakika")
    print(f"Checkpoint           : {CONFIG['pretrain_save']}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history_p1["train_loss"], label="Train Loss", color="#1565C0")
    axes[0].set_title("Faz 1 — PlantCLEF Train Loss")
    axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history_p1["train_acc"], label="Train Acc", color="#2E7D32")
    axes[1].set_title("Faz 1 — PlantCLEF Train Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.suptitle("Botanix Swin-T — PlantCLEF Pre-train History")
    plt.tight_layout()
    plt.savefig("./results/swin_t_pretrain_history.png", dpi=150)
    plt.show()
else:
    print("PlantCLEF bulunamadı — Faz 1 atlandı.")

In [ ]:
# ── FAZ 2: Fine-tune — Multi-label ───────────────────────────────────────────
print("=" * 65)
print("FAZ 2 — Multi-label Fine-tune")
print("=" * 65)

ckpt = Path(CONFIG["pretrain_save"])
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print(f"Pre-train ağırlıkları yüklendi: {ckpt}")
else:
    print("Pre-train checkpoint yok — ImageNet ağırlıklarıyla devam.")

model.replace_head(CONFIG["finetune_classes"])
model = model.to(DEVICE)
model.freeze_backbone()
print(f"Backbone donduruldu (ilk {CONFIG['freeze_epochs']} epoch)")

ft_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
ft_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["finetune_lr"], weight_decay=CONFIG["weight_decay"],
)
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=CONFIG["finetune_epochs"], eta_min=1e-7
)

history = {"train_loss": [], "val_loss": [], "val_acc": [],
           "val_f1_micro": [], "val_f1_macro": []}
best_val_f1 = 0.0
train_start = time.time()

for epoch in range(1, CONFIG["finetune_epochs"] + 1):
    if epoch == CONFIG["freeze_epochs"] + 1:
        model.unfreeze_backbone()
        ft_optimizer = optim.AdamW([
            {"params": model.backbone.parameters(), "lr": CONFIG["finetune_lr"] * 0.05},
            {"params": model.head.parameters(),     "lr": CONFIG["finetune_lr"]},
        ], weight_decay=CONFIG["weight_decay"])
        ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            ft_optimizer,
            T_max=CONFIG["finetune_epochs"] - CONFIG["freeze_epochs"],
            eta_min=1e-7,
        )
        print(f"\nEpoch {epoch}: Backbone açıldı.")

    t0 = time.time()
    tr_loss = train_epoch_ml(model, ft_train_loader, ft_optimizer, ft_criterion, DEVICE)
    vl_loss, vl_f1mi, vl_f1ma = eval_ml(
        model, ft_val_loader, ft_criterion, DEVICE, CONFIG["ml_threshold"]
    )
    lr_now = ft_optimizer.param_groups[0]["lr"]
    ft_scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["val_f1_micro"].append(vl_f1mi)
    history["val_f1_macro"].append(vl_f1ma)

    if vl_f1mi > best_val_f1:
        best_val_f1 = vl_f1mi
        torch.save(model.state_dict(), CONFIG["finetune_save"])

    frozen = "❄" if epoch <= CONFIG["freeze_epochs"] else "🔥"
    print(f"[Fine-tune {frozen}] Epoch [{epoch:02d}/{CONFIG['finetune_epochs']}] "
          f"Loss: {tr_loss:.4f} | Val Loss: {vl_loss:.4f} "
          f"F1-micro: {vl_f1mi:.4f} F1-macro: {vl_f1ma:.4f} | "
          f"LR: {lr_now:.2e} | {time.time()-t0:.1f}s")

total_train_time = time.time() - train_start
print(f"\nFine-tune tamamlandı : {total_train_time/60:.1f} dakika")
print(f"En iyi Val F1-micro  : {best_val_f1:.4f}")

In [ ]:
# ── Eğitim Grafikleri ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train", color="#E53935")
axes[0].plot(history["val_loss"],   label="Val",   color="#1E88E5")
axes[0].axvline(x=CONFIG["freeze_epochs"]-1, color="gray", ls="--", label="Backbone açıldı")
axes[0].set_title("Loss (BCE)"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history["val_f1_micro"], label="F1-micro", color="#43A047")
axes[1].plot(history["val_f1_macro"], label="F1-macro", color="#FB8C00")
axes[1].axvline(x=CONFIG["freeze_epochs"]-1, color="gray", ls="--", label="Backbone açıldı")
axes[1].set_title("Val F1 Score"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle("Botanix Swin-T — Multi-label Fine-tune Training History")
plt.tight_layout()
plt.savefig("./results/swin_t_multilabel_training.png", dpi=150)
plt.show()

In [ ]:
# ── Test Değerlendirmesi ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CONFIG["finetune_save"], map_location=DEVICE))
model.eval()

all_probs_list, all_targets_list = [], []
infer_start = time.time()

with torch.no_grad():
    for imgs, targets in ft_test_loader:
        logits = model(imgs.to(DEVICE))
        all_probs_list.append(torch.sigmoid(logits).cpu())
        all_targets_list.append(targets)

infer_time  = time.time() - infer_start
all_probs   = torch.cat(all_probs_list).numpy()    # [N, 105]
all_targets = torch.cat(all_targets_list).numpy()  # [N, 105]
all_preds   = (all_probs >= CONFIG["ml_threshold"]).astype(float)

# Argmax tabanlı single-label eşdeğer (confusion matrix için)
all_labels_sl = np.argmax(all_targets, axis=1)
all_preds_sl  = np.argmax(all_probs,   axis=1)

f1_micro  = f1_score(all_targets, all_preds, average="micro",    zero_division=0)
f1_macro  = f1_score(all_targets, all_preds, average="macro",    zero_division=0)
f1_samp   = f1_score(all_targets, all_preds, average="samples",  zero_division=0)
prec_mi   = precision_score(all_targets, all_preds, average="micro", zero_division=0)
rec_mi    = recall_score(all_targets, all_preds,    average="micro", zero_division=0)
mAP       = average_precision_score(all_targets, all_probs, average="macro")

print(f"\n{'='*55}")
print(f"Test Sonuçları — Multi-label (threshold={CONFIG['ml_threshold']})")
print(f"{'='*55}")
print(f"F1-micro   : {f1_micro:.4f}")
print(f"F1-macro   : {f1_macro:.4f}")
print(f"F1-samples : {f1_samp:.4f}")
print(f"Precision  : {prec_mi:.4f}")
print(f"Recall     : {rec_mi:.4f}")
print(f"mAP        : {mAP:.4f}")
print(f"Inference  : {infer_time/len(ft_test_raw)*1000:.2f} ms/görüntü")

In [ ]:
# ── Kapsamlı Grafiksel Değerlendirme ─────────────────────────────────────────
MODEL_LABEL  = "Botanix Swin-T (PlantCLEF → Multi-label)"
MODEL_PREFIX = "swin_t_multilabel"

# ── 1. Eğitim Grafikleri (Loss + Accuracy) ── (zaten üstte çizildi)

# ── 2. Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels_sl, all_preds_sl)
plt.figure(figsize=(22, 20))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=False, yticklabels=False)
plt.title(f"{MODEL_LABEL} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_confusion_matrix.png", dpi=150)
plt.show()

# ── 3. Per-Class F1 Score ─────────────────────────────────────────────────────
per_class_f1 = f1_score(all_targets, all_preds, average=None, zero_division=0)
sorted_idx   = np.argsort(per_class_f1)
worst20 = [(class_names[i], per_class_f1[i]) for i in sorted_idx[:20]]
best20  = [(class_names[i], per_class_f1[i]) for i in sorted_idx[-20:][::-1]]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].barh([x[0][:25] for x in worst20], [x[1] for x in worst20], color="#EF5350")
axes[0].set_title(f"{MODEL_LABEL} — En Düşük F1 (20 Sınıf)")
axes[0].set_xlabel("F1 Score"); axes[0].set_xlim(0, 1)
axes[1].barh([x[0][:25] for x in best20],  [x[1] for x in best20],  color="#66BB6A")
axes[1].set_title(f"{MODEL_LABEL} — En Yüksek F1 (20 Sınıf)")
axes[1].set_xlabel("F1 Score"); axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_per_class_f1.png", dpi=150)
plt.show()

# ── 4. Top-10 Karıştırılan Sınıf Çiftleri ────────────────────────────────────
cm_err = confusion_matrix(all_labels_sl, all_preds_sl)
np.fill_diagonal(cm_err, 0)
flat_idx = np.argsort(cm_err.ravel())[::-1][:10]
rows_e, cols_e = np.unravel_index(flat_idx, cm_err.shape)

fig, ax = plt.subplots(figsize=(12, 5))
labels_top = [f"{class_names[r][:15]}→{class_names[c][:15]}" for r, c in zip(rows_e, cols_e)]
counts_top = [cm_err[r, c] for r, c in zip(rows_e, cols_e)]
bars = ax.barh(labels_top[::-1], counts_top[::-1], color="#5C6BC0")
for bar, val in zip(bars, counts_top[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=9)
ax.set_title(f"{MODEL_LABEL} — En Çok Karıştırılan 10 Sınıf Çifti")
ax.set_xlabel("Yanlış Sınıflandırma Sayısı")
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_top_errors.png", dpi=150)
plt.show()

# ── 5. Makro-Ortalama ROC Eğrisi ─────────────────────────────────────────────
fpr_dict, tpr_dict, roc_dict = {}, {}, {}
for i in range(CONFIG["finetune_classes"]):
    if all_targets[:, i].sum() > 0:
        fpr_dict[i], tpr_dict[i], _ = roc_curve(all_targets[:, i], all_probs[:, i])
        roc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

all_fpr  = np.unique(np.concatenate([fpr_dict[i] for i in roc_dict]))
mean_tpr = np.zeros_like(all_fpr)
for i in roc_dict:
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= len(roc_dict)
macro_auc = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(all_fpr, mean_tpr, color="#1565C0", lw=2,
        label=f"Makro-Ortalama ROC (AUC = {macro_auc:.4f})")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL_LABEL} — ROC Eğrisi (Makro Ortalama)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_roc.png", dpi=150)
plt.show()
print(f"Makro-Ortalama ROC AUC: {macro_auc:.4f}")
print("Grafikler ./results/ dizinine kaydedildi.")

In [ ]:
# ── Bilinmeyen Hastalık Stratejisi ────────────────────────────────────────────
# Problem: Model bir bitkiyi tanıyor ama hastalığını bilmiyorsa ne döner?
# Çözüm: 3 katmanlı strateji
#   1. Confidence Threshold : max_prob < unknown_threshold → "Unknown Disease"
#   2. Entropy Uncertainty  : normalize entropy > entropy_threshold → "Uncertain"
#   3. Healthy kontrolü     : sınıf adında 'healthy' varsa önce kontrol et

def predict_with_uncertainty(
    probs: np.ndarray,
    class_names: list,
    ml_threshold: float = 0.5,
    unknown_threshold: float = 0.3,
    entropy_threshold: float = 0.85,
):
    """
    probs: [num_classes] — sigmoid çıkışı
    Dönüş: (detected_classes, status, max_prob, entropy)
    status: 'detected' | 'unknown' | 'uncertain' | 'healthy'
    """
    max_prob = probs.max()

    # Normalize entropy (0=emin, 1=tamamen belirsiz)
    eps = 1e-8
    p_clip = np.clip(probs, eps, 1 - eps)
    entropy = -np.sum(p_clip * np.log(p_clip) + (1-p_clip) * np.log(1-p_clip))
    max_entropy = len(probs) * np.log(2)
    norm_entropy = entropy / max_entropy

    # Healthy sınıfı kontrolü
    healthy_idxs = [i for i, n in enumerate(class_names) if "healthy" in n.lower()]
    is_healthy = any(probs[i] >= ml_threshold for i in healthy_idxs)

    if is_healthy:
        healthy_names = [class_names[i] for i in healthy_idxs if probs[i] >= ml_threshold]
        return healthy_names, "healthy", float(max_prob), float(norm_entropy)

    if max_prob < unknown_threshold:
        return ["Unknown Disease"], "unknown", float(max_prob), float(norm_entropy)

    if norm_entropy > entropy_threshold:
        return ["Uncertain"], "uncertain", float(max_prob), float(norm_entropy)

    detected = [class_names[i] for i, p in enumerate(probs) if p >= ml_threshold]
    if not detected:
        # Eşiğin altında ama max_prob yeterli — en yüksek olasılıklıyı al
        detected = [class_names[int(np.argmax(probs))]]
    return detected, "detected", float(max_prob), float(norm_entropy)

# ── Bilinmeyen Hastalık Simülasyonu ──────────────────────────────────────────
# Test seti üzerinde 3 senaryoyu göster
np.random.seed(42)
sample_idxs = np.random.choice(len(all_probs), 10, replace=False)

print(f"{'='*70}")
print(f"Bilinmeyen Hastalık Stratejisi — Örnek Tahminler")
print(f"{'='*70}")
statuses = []
for idx in sample_idxs:
    probs_i  = all_probs[idx]
    true_lbl = class_names[int(all_labels_sl[idx])]
    detected, status, max_p, entr = predict_with_uncertainty(
        probs_i, class_names,
        CONFIG["ml_threshold"],
        CONFIG["unknown_threshold"],
        CONFIG["entropy_threshold"],
    )
    statuses.append(status)
    print(f"Gerçek: {true_lbl[:30]:<30} | "
          f"Tahmin: {str(detected[0])[:25]:<25} | "
          f"Status: {status:<10} | "
          f"MaxP: {max_p:.3f} | Entropy: {entr:.3f}")

# ── Tüm test seti üzerinde status dağılımı ────────────────────────────────────
all_statuses = []
for i in range(len(all_probs)):
    _, status, _, _ = predict_with_uncertainty(
        all_probs[i], class_names,
        CONFIG["ml_threshold"],
        CONFIG["unknown_threshold"],
        CONFIG["entropy_threshold"],
    )
    all_statuses.append(status)

from collections import Counter
status_counts = Counter(all_statuses)
print(f"\n{'='*50}")
print("Test Seti — Tahmin Durum Dağılımı")
print(f"{'='*50}")
for s, c in status_counts.items():
    print(f"  {s:<12}: {c:4d} ({c/len(all_probs)*100:.1f}%)")

# ── Status Dağılımı Grafiği ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors_map = {"detected": "#43A047", "healthy": "#1E88E5",
              "unknown": "#E53935",  "uncertain": "#FB8C00"}
labels_s = list(status_counts.keys())
vals_s   = list(status_counts.values())
cols_s   = [colors_map.get(l, "#9E9E9E") for l in labels_s]
bars = ax.bar(labels_s, vals_s, color=cols_s, edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, vals_s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{val}\n({val/len(all_probs)*100:.1f}%)",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title(f"{MODEL_LABEL} — Bilinmeyen Hastalık Strateji Dağılımı")
ax.set_ylabel("Görüntü Sayısı"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_unknown_strategy.png", dpi=150)
plt.show()

In [ ]:
# ── Sonuçları Kaydet ──────────────────────────────────────────────────────────
results = {
    "model":           "Botanix Swin-T (PlantCLEF Pre-train → Multi-label)",
    "task":            "multi-label",
    "f1_micro":        round(float(f1_micro),  4),
    "f1_macro":        round(float(f1_macro),  4),
    "f1_samples":      round(float(f1_samp),   4),
    "precision":       round(float(prec_mi),   4),
    "recall":          round(float(rec_mi),    4),
    "mAP":             round(float(mAP),       4),
    "roc_auc_macro":   round(float(macro_auc), 4),
    "ml_threshold":    CONFIG["ml_threshold"],
    "unknown_strategy": {
        "unknown_threshold": CONFIG["unknown_threshold"],
        "entropy_threshold": CONFIG["entropy_threshold"],
        "status_distribution": dict(status_counts),
    },
    "pretrain_time_min":  round(total_pretrain_time / 60, 2),
    "finetune_time_min":  round(total_train_time / 60, 2),
    "inference_ms_per_image": round(infer_time / len(ft_test_raw) * 1000, 4),
    "best_val_f1_micro": round(float(best_val_f1), 4),
    "total_params":      total_params,
    "config":            CONFIG,
}
with open(f"./results/{MODEL_PREFIX}_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## Mobil Uyumluluk Notu — Botanix Swin-T Multi-label

| Özellik | Değer |
|---------|-------|
| Mimari | Swin Transformer Tiny |
| Parametre sayısı | ~28M |
| Model boyutu (.pth) | ~110 MB |
| Model boyutu (INT8) | ~28 MB |
| GPU Inference | ~5-8 ms/görüntü |
| Mobil hedef | ~15-30 ms |
| Mobil uygunluğu | ✅ Orta-Yüksek |
| Çıkış | Multi-label sigmoid (105 sınıf) |

**Bilinmeyen Hastalık Yanıtı:**
```python
probs = torch.sigmoid(model(image))  # [1, 105]
detected, status, max_p, entropy = predict_with_uncertainty(
    probs.numpy(), class_names,
    ml_threshold=0.5,
    unknown_threshold=0.3,
    entropy_threshold=0.85,
)
# status: 'detected' | 'healthy' | 'unknown' | 'uncertain'
```

**INT8 Quantization:**
```python
model_int8 = torch.quantization.quantize_dynamic(
    model.cpu(), {torch.nn.Linear}, dtype=torch.qint8
)
torch.jit.save(torch.jit.script(model_int8), "swin_t_int8.pt")
```